![Identidade visual da disciplina](../capa-analise-visualizacao-python-ia.png)

# IA001 — Análise e Visualização de Dados com Python e Ferramentas Assistidas por IA

## Da fonte ao mapa: obtenção e armazenamento de dados geográficos

**Objetivo:** compreender os arquivos usados nos mapas de Porto Alegre e reproduzir um fluxo de download, recorte, junção e exportação para outras cidades brasileiras.

**Produto:** GeoPackage para análise, GeoJSON para mapas web, CSV de indicadores e um arquivo JSON com as fontes e os parâmetros utilizados.

**Percurso:** (A) reconstruir o caso de Porto Alegre; (B) escolher Porto Alegre, Recife, Rio de Janeiro ou São Paulo; (C) preparar, salvar, reabrir e visualizar os dados.

Os links foram conferidos em **22/09/2026**. A primeira execução precisa de internet. Nas seguintes, os downloads existentes são reutilizados. Os mapas web continuam dependendo de recursos externos.

## 1. Geometria e indicadores são dados diferentes

Um mapa coroplético combina uma **malha**, com os polígonos das unidades territoriais, e uma **tabela**, com os indicadores dessas unidades. O nome de uma cidade ou as coordenadas de seu centro não fornecem seus limites internos.

| Arquivo | O que armazena | Uso neste roteiro |
|---|---|---|
| GeoPackage (`.gpkg`) | Geometrias, atributos e sistema de referência, em um arquivo | Manter a malha e os dados preparados |
| CSV (`.csv`) | Tabela; neste caso, sem geometria | População e indicadores |
| GeoJSON (`.geojson`) | Geometrias e atributos em JSON | Mapas web, em longitude/latitude |
| ZIP (`.zip`) | Arquivos compactados | Distribuição das tabelas nacionais |

Um Shapefile costuma depender de vários arquivos (`.shp`, `.shx`, `.dbf`, `.prj` etc.). Aqui usamos GeoPackage para simplificar o armazenamento.

## 2. Preparação e pastas

O código funciona a partir da raiz `IA001` ou da pasta `notebooks`. As saídas novas ficam em `dados/aquisicao_geografica`, sem substituir os dados dos notebooks anteriores.

```text
dados/
├── porto_alegre/                  # arquivos já usados em aula
└── aquisicao_geografica/
    ├── brutos/                   # cópias baixadas das fontes
    └── preparados/<cidade>/      # resultados deste roteiro
```

Instale as dependências no mesmo kernel usado para executar o notebook.

In [20]:
# Se necessário, descomente:
# %pip install geopandas pandas folium

from pathlib import Path
from datetime import datetime, timezone
from urllib.request import urlopen, Request
import hashlib
import json
import re
import unicodedata
import zipfile

import geopandas as gpd
import pandas as pd
import folium
from IPython.display import display, FileLink

PASTA_NOTEBOOKS = Path("notebooks") if Path("notebooks/dados").exists() else Path(".")
if not (PASTA_NOTEBOOKS / "dados").exists():
    raise FileNotFoundError("Abra o notebook a partir da raiz IA001 ou da pasta notebooks.")
PASTA_TRABALHO = PASTA_NOTEBOOKS / "dados" / "aquisicao_geografica"
PASTA_BRUTOS = PASTA_TRABALHO / "brutos"
PASTA_BRUTOS.mkdir(parents=True, exist_ok=True)
print("GeoPandas:", gpd.__version__, "| pandas:", pd.__version__)
print("Pasta de trabalho:", PASTA_TRABALHO.resolve())

GeoPandas: 1.0.1 | pandas: 2.3.3
Pasta de trabalho: /Users/valandro/Downloads/aula03_notebooks/dados/aquisicao_geografica


### Download com reutilização e registro de procedência

`baixar` grava os bytes recebidos e registra URL, data UTC, tamanho e SHA-256. O hash identifica a cópia exata; não prova, sozinho, a qualidade do dado. Um arquivo existente com registro é conferido antes de ser reutilizado.

O download passa por um arquivo `.part`; só depois de concluído ganha o nome final. Se a fonte mudar de endereço, procure o arquivo na página oficial e atualize a URL. Não contorne erros de certificado desativando HTTPS.

Um arquivo preexistente sem registro pode ser lido, mas sua data de download permanece desconhecida. Para uma atualização deliberada, use uma nova pasta de brutos, preservando a versão anterior.

In [21]:
def sha256(arquivo):
    resumo = hashlib.sha256()
    with Path(arquivo).open("rb") as f:
        for bloco in iter(lambda: f.read(1024 * 1024), b""):
            resumo.update(bloco)
    return resumo.hexdigest()

def baixar(url, destino):
    destino = Path(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    registro = destino.with_suffix(destino.suffix + ".fonte.json")
    if destino.exists():
        if registro.exists():
            meta = json.loads(registro.read_text(encoding="utf-8"))
            if meta["url"] != url or meta["sha256"] != sha256(destino):
                raise ValueError(f"Cache incompatível ou modificado: {destino}")
        print("Reutilizando:", destino.name)
        return destino
    temporario = destino.with_suffix(destino.suffix + ".part")
    try:
        with urlopen(Request(url, headers={"User-Agent": "IA001-aula-geografia"}), timeout=120) as resposta:
            url_final = resposta.geturl()
            with temporario.open("wb") as f:
                for bloco in iter(lambda: resposta.read(1024 * 1024), b""):
                    f.write(bloco)
        if temporario.stat().st_size == 0:
            raise ValueError("A fonte devolveu um arquivo vazio.")
        temporario.replace(destino)
    finally:
        if temporario.exists():
            temporario.unlink()
    meta = {"url": url, "url_final": url_final,
            "baixado_em_utc": datetime.now(timezone.utc).isoformat(),
            "bytes": destino.stat().st_size, "sha256": sha256(destino)}
    registro.write_text(json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    print("Baixado:", destino.name, "—", meta["bytes"], "bytes")
    return destino

## 3. Como os arquivos de Porto Alegre foram obtidos

O handoff anterior informa as fontes, mas não registra o comando de download original. Esta seção **reconstrói um procedimento reproduzível**: em 22/09/2026, os downloads dos links abaixo foram comparados byte a byte com os arquivos da aula, e eram idênticos.

1. No [diretório de bairros do IBGE](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/gpkg/UF/RS/), baixar `RS_bairros_CD2022.gpkg`. O arquivo contém bairros de vários municípios do **Rio Grande do Sul**, não apenas de Porto Alegre.
2. Em [ObservaPOA — Mapas](https://prefeitura.poa.br/smpg/observapoa/mapas), escolher **População por Bairro — Universo — Censo 2022 → CSV**.
3. Salvar o CSV publicado como `Pop_Bairro_Censo_2022.csv` com o nome local `populacao_bairros_porto_alegre_censo2022.csv`. A mudança foi de nome, não de conteúdo.

Os dados originais foram armazenados; o recorte municipal, a limpeza e a junção são realizados posteriormente pelo código. A célula a seguir reproduz o download em uma nova pasta.

In [22]:
BASE_MALHAS = (
    "https://geoftp.ibge.gov.br/organizacao_do_territorio/"
    "malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/"
    "censo_2022/"
)
URL_RS = BASE_MALHAS + "bairros/gpkg/UF/RS/RS_bairros_CD2022.gpkg"
URL_POA = (
    "https://prefeitura.poa.br/sites/default/files/usu_doc/hotsites/"
    "smpae/observapoa/Pop_Bairro_Censo_2022.csv"
)
ARQUIVO_RS = baixar(URL_RS, PASTA_BRUTOS / "RS_bairros_CD2022.gpkg")
ARQUIVO_POA = baixar(URL_POA, PASTA_BRUTOS / "populacao_bairros_porto_alegre_censo2022.csv")

# Se as cópias anteriores estiverem presentes, compare o conteúdo.
for arquivo in [ARQUIVO_RS, ARQUIVO_POA]:
    anterior = PASTA_NOTEBOOKS / "dados" / "porto_alegre" / arquivo.name
    if anterior.exists():
        print(arquivo.name, "— idêntico à cópia anterior:", sha256(arquivo) == sha256(anterior))

Reutilizando: RS_bairros_CD2022.gpkg
Reutilizando: populacao_bairros_porto_alegre_censo2022.csv


### Inspecionar antes de transformar

`CD_MUN` identifica o município; `CD_BAIRRO` identifica o bairro. Preserve os códigos como texto. `EPSG:4674` identifica SIRGAS 2000 em coordenadas geográficas; não é uma projeção métrica.

Um bairro pode aparecer em várias linhas por estar dividido em partes territoriais. `dissolve` reúne essas geometrias por código e nome. Não some uma população repetida em cada fragmento.

In [23]:
malha_rs = gpd.read_file(ARQUIVO_RS)
print("CRS original:", malha_rs.crs)
print("Colunas:", malha_rs.columns.tolist())
poa = malha_rs.loc[
    malha_rs["CD_MUN"].astype(str) == "4314902",
    ["CD_BAIRRO", "NM_BAIRRO", "geometry"]
].copy()
print("Fragmentos:", len(poa), "| códigos de bairro:", poa["CD_BAIRRO"].nunique())
poa = poa.dissolve(by=["CD_BAIRRO", "NM_BAIRRO"], as_index=False)
display(poa.drop(columns="geometry").head())

CRS original: EPSG:4674
Colunas: ['CD_REGIAO', 'NM_REGIAO', 'CD_UF', 'NM_UF', 'CD_MUN', 'NM_MUN', 'CD_DIST', 'NM_DIST', 'CD_SUBDIST', 'NM_SUBDIST', 'CD_BAIRRO', 'NM_BAIRRO', 'CD_RGINT', 'NM_RGINT', 'CD_RGI', 'NM_RGI', 'CD_CONCURB', 'NM_CONCURB', 'geometry']
Fragmentos: 99 | códigos de bairro: 94


,CD_BAIRRO,NM_BAIRRO
0,4314902001,Medianeira
1,4314902002,Praia de Belas
2,4314902003,Cidade Baixa
3,4314902004,Menino-Deus
4,4314902005,Farroupilha


### Ler a tabela do ObservaPOA

O CSV possui uma linha de título antes do cabeçalho, separador `;`, codificação Latin-1 e ponto como separador de milhares. Esses parâmetros foram identificados neste arquivo; não são uma regra universal para CSV brasileiros.

A linha `Total` não é um bairro. Nesta tabela os nomes precisam ser normalizados para a junção, pois não há código IBGE de bairro. A normalização não resolve nomes historicamente diferentes nem mudanças de limites.

In [24]:
print("Primeiras linhas do arquivo:")
print("\n".join(ARQUIVO_POA.read_text(encoding="latin1").splitlines()[:5]))

pop_poa = pd.read_csv(
    ARQUIVO_POA, sep=";", skiprows=1, encoding="latin1",
    usecols=["Bairro", "Número"], dtype=str
).dropna(subset=["Bairro", "Número"])
pop_poa = pop_poa.loc[pop_poa["Bairro"] != "Total"].copy()
pop_poa["populacao_observapoa"] = pop_poa["Número"].str.replace(".", "", regex=False).astype(int)

def normalizar_nome(texto):
    texto = unicodedata.normalize("NFKD", str(texto).replace("’", "'"))
    texto = texto.encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", " ", texto.lower()).strip()

poa["chave_nome"] = poa["NM_BAIRRO"].map(normalizar_nome)
pop_poa["chave_nome"] = pop_poa["Bairro"].map(normalizar_nome)
poa_historico = poa.merge(
    pop_poa[["chave_nome", "populacao_observapoa"]],
    on="chave_nome", how="left", validate="one_to_one", indicator=True
)
print(poa_historico["_merge"].value_counts().to_string())
if not poa_historico["_merge"].eq("both").all():
    raise ValueError("Revise a correspondência dos nomes antes de produzir o mapa.")
print("Nomes da tabela sem geometria:", sorted(set(pop_poa.chave_nome) - set(poa.chave_nome)))
display(poa_historico[["CD_BAIRRO", "NM_BAIRRO", "populacao_observapoa"]].head())

Primeiras linhas do arquivo:
População residente, por bairros, de Porto Alegre - Censo 2022;;;
Bairro;Número;Percentual;
Restinga;62.448;4,69;
Lomba do Pinheiro;59.200;4,44;
Sarandi;51.539;3,87;
_merge
both          94
left_only      0
right_only     0
Nomes da tabela sem geometria: []


,CD_BAIRRO,NM_BAIRRO,populacao_observapoa
0,4314902001,Medianeira,8749
1,4314902002,Praia de Belas,1522
2,4314902003,Cidade Baixa,13014
3,4314902004,Menino-Deus,27961
4,4314902005,Farroupilha,774


## 4. Adaptar para outra cidade: escolha a unidade territorial

Para generalizar, usaremos **malha e tabela de população do IBGE**, unidas por código. Este é um segundo fluxo: ele não substitui silenciosamente o CSV do ObservaPOA nos notebooks anteriores.

| Cidade | UF | Código do município | Unidade utilizada |
|---|---|---|---|
| Porto Alegre | RS | 4314902 | Bairro |
| Recife | PE | 2611606 | Bairro |
| Rio de Janeiro | RJ | 3304557 | Bairro |
| São Paulo | SP | 3550308 | Distrito |

Na malha consultada do Censo 2022, o município de São Paulo não possui registros na camada de bairros. Seus 96 distritos são uma escolha adequada para este exemplo, conforme a [divisão administrativa municipal](https://prefeitura.sp.gov.br/web/licenciamento/w/servicos/341586). Não os renomeie como bairros.

Para outras cidades, confirme a unidade disponível, o código IBGE e a correspondência entre os limites e o ano da estatística. As unidades do Censo 2022 não necessariamente representam divisões administrativas atuais. Ausência de bairros não significa ausência de população.

**Para experimentar outra cidade:** altere `CIDADE` abaixo e execute novamente a partir desta célula. Para executar tudo novamente, reinicie o kernel e use *Run All*.

In [25]:
CIDADES = {
    "porto_alegre": {"nome": "Porto Alegre", "uf": "RS", "codigo": "4314902", "nivel": "bairros"},
    "recife": {"nome": "Recife", "uf": "PE", "codigo": "2611606", "nivel": "bairros"},
    "rio_de_janeiro": {"nome": "Rio de Janeiro", "uf": "RJ", "codigo": "3304557", "nivel": "bairros"},
    "sao_paulo": {"nome": "São Paulo", "uf": "SP", "codigo": "3550308", "nivel": "distritos"},
}
CIDADE = "porto_alegre"  # troque por "recife", "rio_de_janeiro" ou "sao_paulo"
config = CIDADES[CIDADE]
NIVEL = config["nivel"]
CODIGO = "CD_BAIRRO" if NIVEL == "bairros" else "CD_DIST"
NOME = "NM_BAIRRO" if NIVEL == "bairros" else "NM_DIST"
print(config)

{'nome': 'Porto Alegre', 'uf': 'RS', 'codigo': '4314902', 'nivel': 'bairros'}


## 5. Baixar a malha estadual e os agregados de população

As malhas são distribuídas por UF; as tabelas básicas utilizadas aqui cobrem o Brasil. Em ambos os casos faremos o recorte pelo município.

A versão das tabelas está explícita na URL: `20260520`. A data é de atualização do arquivo, não o ano da população, que continua sendo **2022**. Consulte o [diretório de agregados do IBGE](https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/) se o endereço mudar.

No tema **Básico**, `v0001` corresponde ao total de pessoas. Confira o [dicionário de dados dessa versão](https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/dicionario_de_dados_agregados_por_setores_censitarios_20260520.xlsx) antes de selecionar outras variáveis. Não confunda população com domicílios.

In [26]:
BASE_TABELAS = (
    "https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/"
    "Agregados_por_Setores_Censitarios/"
)
PASTAS_TABELAS = {"bairros": "Agregados_por_Bairro_csv", "distritos": "Agregados_por_Distrito_csv"}
nome_malha = f"{config['uf']}_{NIVEL}_CD2022.gpkg"
nome_zip = f"Agregados_por_{NIVEL}_basico_BR_20260520.zip"
URL_MALHA = BASE_MALHAS + f"{NIVEL}/gpkg/UF/{config['uf']}/{nome_malha}"
URL_TABELA = BASE_TABELAS + f"{PASTAS_TABELAS[NIVEL]}/{nome_zip}"
ARQUIVO_MALHA = baixar(URL_MALHA, PASTA_BRUTOS / nome_malha)
ARQUIVO_TABELA = baixar(URL_TABELA, PASTA_BRUTOS / nome_zip)
print("Malha:", URL_MALHA)
print("Tabela:", URL_TABELA)

Reutilizando: RS_bairros_CD2022.gpkg
Reutilizando: Agregados_por_bairros_basico_BR_20260520.zip
Malha: https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/gpkg/UF/RS/RS_bairros_CD2022.gpkg
Tabela: https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/Agregados_por_Bairro_csv/Agregados_por_bairros_basico_BR_20260520.zip


## 6. Recortar e reunir geometrias

Selecionamos o município pelo código, preservando apenas a chave, o nome e a geometria. Uma seleção vazia deve interromper o processo: ela pode indicar unidade territorial indisponível ou código incorreto.

A tabela de população será unida **depois** de reunir os fragmentos geométricos. Isso evita duplicar habitantes ao somar registros de um mesmo bairro.

In [27]:
malha = gpd.read_file(ARQUIVO_MALHA)
territorios = malha.loc[
    malha["CD_MUN"].astype(str) == config["codigo"],
    [CODIGO, NOME, "geometry"]
].copy()
if territorios.empty:
    raise ValueError("Município sem registros nessa malha. Confira código e unidade territorial.")
if territorios.crs is None:
    raise ValueError("A malha não informa o sistema de referência.")
territorios[CODIGO] = territorios[CODIGO].astype(str)
fragmentos = len(territorios)
territorios = territorios.dissolve(by=[CODIGO, NOME], as_index=False)
if territorios[CODIGO].duplicated().any():
    raise ValueError("Há códigos com nomes divergentes; revise a malha.")
if territorios.geometry.is_empty.any() or territorios.geometry.isna().any() or not territorios.geometry.is_valid.all():
    raise ValueError("Há geometrias vazias ou inválidas; inspecione antes de continuar.")
print(f"{config['nome']}: {fragmentos} fragmentos → {len(territorios)} {NIVEL}")
print("CRS:", territorios.crs)

Porto Alegre: 99 fragmentos → 94 bairros
CRS: EPSG:4674


## 7. Ler o CSV de dentro do ZIP e unir por código

Não é necessário extrair todos os arquivos: `ZipFile.open` fornece o CSV para `read_csv`. Os códigos são lidos como texto; o campo de população é convertido depois.

Verificamos chaves duplicadas, ausências e códigos presentes em apenas uma das fontes. Um valor ausente não deve ser convertido automaticamente em zero. A validação `one_to_one` impede junções que multipliquem linhas.

In [28]:
with zipfile.ZipFile(ARQUIVO_TABELA) as pacote:
    csvs = [nome for nome in pacote.namelist() if nome.lower().endswith(".csv")]
    if len(csvs) != 1:
        raise ValueError(f"Selecione explicitamente o CSV correto: {csvs}")
    with pacote.open(csvs[0]) as f:
        tabela = pd.read_csv(f, sep=";", encoding="latin1", dtype=str)

tabela_cidade = tabela.loc[
    tabela["CD_MUN"] == config["codigo"], [CODIGO, "v0001"]
].copy()
if tabela_cidade.empty or tabela_cidade[CODIGO].duplicated().any():
    raise ValueError("Tabela vazia ou com códigos repetidos.")
tabela_cidade["populacao"] = pd.to_numeric(tabela_cidade["v0001"], errors="coerce")
if tabela_cidade["populacao"].isna().any() or tabela_cidade["populacao"].lt(0).any():
    raise ValueError("Há população ausente, suprimida ou inválida. Consulte o dicionário.")

sem_tabela = set(territorios[CODIGO]) - set(tabela_cidade[CODIGO])
sem_geometria = set(tabela_cidade[CODIGO]) - set(territorios[CODIGO])
print("Geometrias sem tabela:", sorted(sem_tabela))
print("Registros sem geometria:", sorted(sem_geometria))
if sem_tabela or sem_geometria:
    raise ValueError("As chaves não coincidem; verifique versões e recortes.")

analise = territorios.merge(
    tabela_cidade[[CODIGO, "populacao"]], on=CODIGO, how="left", validate="one_to_one"
)
analise["populacao"] = analise["populacao"].astype("int64")
print("População nas unidades selecionadas:", f"{analise.populacao.sum():,}")
display(analise.drop(columns="geometry").head())

Geometrias sem tabela: []
Registros sem geometria: []
População nas unidades selecionadas: 1,332,845


,CD_BAIRRO,NM_BAIRRO,populacao
0,4314902001,Medianeira,8749
1,4314902002,Praia de Belas,1522
2,4314902003,Cidade Baixa,13014
3,4314902004,Menino-Deus,27961
4,4314902005,Farroupilha,774


### Comparar fontes no caso de Porto Alegre

Se Porto Alegre estiver selecionada, a próxima célula compara os valores do ObservaPOA com os agregados do IBGE. Diferenças devem motivar investigação das versões e dos limites, não uma correção arbitrária.

Mesmo uma junção completa não garante que os bairros cubram todo o município. Para concluir sobre cobertura, compare também com a malha municipal e com o total oficial da população; não confunda a soma das áreas dos bairros com a área oficial do município.

In [29]:
if CIDADE == "porto_alegre":
    comparacao = analise[[CODIGO, NOME, "populacao"]].merge(
        poa_historico[["CD_BAIRRO", "populacao_observapoa"]],
        on="CD_BAIRRO", validate="one_to_one"
    )
    comparacao["diferenca"] = comparacao["populacao"] - comparacao["populacao_observapoa"]
    print("Bairros com valores diferentes:", int(comparacao.diferenca.ne(0).sum()))
    display(comparacao.loc[comparacao.diferenca.ne(0)])
else:
    print("Comparação com ObservaPOA aplicável apenas a Porto Alegre.")

Bairros com valores diferentes: 0


,CD_BAIRRO,NM_BAIRRO,populacao,populacao_observapoa,diferenca


## 8. Preparar área e densidade

Para estas cidades, escolhemos automaticamente uma projeção UTM local usando `estimate_utm_crs()`. Isso evita reutilizar a zona de Porto Alegre em Recife ou no Rio. A projeção estimada é registrada nas saídas.

Para territórios muito extensos ou que atravessam várias zonas, avalie uma projeção apropriada à análise em vez de aplicar esta escolha automaticamente.

Calculamos a área antes de simplificar as geometrias. A densidade é população dividida por área, em habitantes por km².

In [30]:
CRS_AREA = analise.estimate_utm_crs()
if CRS_AREA is None:
    raise ValueError("Não foi possível estimar a projeção; escolha um CRS métrico adequado.")
analise["area_km2"] = analise.to_crs(CRS_AREA).area / 1_000_000
if not analise["area_km2"].gt(0).all():
    raise ValueError("Há áreas inválidas.")
analise["densidade_hab_km2"] = analise["populacao"] / analise["area_km2"]
print("Projeção para cálculo de área:", CRS_AREA)
display(analise.drop(columns="geometry").head())

Projeção para cálculo de área: EPSG:32722


,CD_BAIRRO,NM_BAIRRO,populacao,area_km2,densidade_hab_km2
0,4314902001,Medianeira,8749,1.397104,6262.240551
1,4314902002,Praia de Belas,1522,2.615000,582.026766
2,4314902003,Cidade Baixa,13014,0.759417,17136.838796
3,4314902004,Menino-Deus,27961,2.298086,12167.082680
4,4314902005,Farroupilha,774,0.600636,1288.633596


## 9. Salvar os dados preparados

O GeoPackage conserva a geometria sem simplificação. O CSV contém somente os atributos e usa UTF-8. O GeoJSON usa `EPSG:4326`, em longitude/latitude, e uma simplificação de 25 m para reduzir o tamanho do mapa web.

`to_crs` **transforma** coordenadas; `set_crs` apenas declara seu sistema de referência. Trocar o rótulo do CRS não reprojeta os dados.

Esta célula atualiza somente os arquivos preparados da cidade selecionada. Os brutos permanecem intactos.

In [31]:
PASTA_SAIDA = PASTA_TRABALHO / "preparados" / CIDADE
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
SAIDA_GPKG = PASTA_SAIDA / "territorios_indicadores.gpkg"
SAIDA_CSV = PASTA_SAIDA / "indicadores.csv"
SAIDA_GEOJSON = PASTA_SAIDA / "territorios_web.geojson"

analise.to_file(SAIDA_GPKG, layer="territorios", driver="GPKG", mode="w", index=False)
analise.drop(columns="geometry").to_csv(SAIDA_CSV, index=False, encoding="utf-8")
web = analise.to_crs(CRS_AREA).copy()
web["geometry"] = web.geometry.simplify(25, preserve_topology=True)
web = web.to_crs(4326)
web.to_file(SAIDA_GEOJSON, driver="GeoJSON", index=False)
for arquivo in [SAIDA_GPKG, SAIDA_CSV, SAIDA_GEOJSON]:
    print(arquivo.name, "—", round(arquivo.stat().st_size / 1024), "KiB")

territorios_indicadores.gpkg — 504 KiB
indicadores.csv — 6 KiB
territorios_web.geojson — 155 KiB


### Registrar fontes e decisões

Um nome de arquivo não informa qual edição foi usada nem como o indicador foi calculado. O registro JSON abaixo acompanha os dados e distingue o ano de referência da data de preparação.

Ao entregar o trabalho, inclua esse registro. Ele permite reconhecer a cópia usada mesmo que o órgão atualize seu arquivo posteriormente.

In [32]:
def descrever_fonte(arquivo, url):
    registro = arquivo.with_suffix(arquivo.suffix + ".fonte.json")
    meta = json.loads(registro.read_text(encoding="utf-8")) if registro.exists() else {}
    return {"arquivo": arquivo.name, "url": url, "sha256": sha256(arquivo),
            "bytes": arquivo.stat().st_size, "baixado_em_utc": meta.get("baixado_em_utc"),
            "observacao": "Data desconhecida se a cópia já existia sem registro."}

metadados = {
    "cidade": config, "ano_populacao": 2022, "ano_malha": 2022,
    "edicao_tabela": "20260520", "variavel_populacao": "v0001 — Total de pessoas",
    "preparado_em_utc": datetime.now(timezone.utc).isoformat(),
    "fontes": [descrever_fonte(ARQUIVO_MALHA, URL_MALHA), descrever_fonte(ARQUIVO_TABELA, URL_TABELA)],
    "chave_juncao": CODIGO, "unidades": len(analise),
    "populacao_unidades": int(analise.populacao.sum()),
    "crs_original": str(analise.crs), "crs_area": str(CRS_AREA), "crs_web": "EPSG:4326",
    "densidade": "populacao / area_km2", "simplificacao_web_m": 25,
    "transformacoes": ["filtro por CD_MUN", "dissolve por código e nome", "junção por código"],
    "versoes": {"geopandas": gpd.__version__, "pandas": pd.__version__, "folium": folium.__version__},
    "saidas": {p.name: {"sha256": sha256(p), "bytes": p.stat().st_size}
               for p in [SAIDA_GPKG, SAIDA_CSV, SAIDA_GEOJSON]},
}
SAIDA_FONTES = PASTA_SAIDA / "fontes_e_preparo.json"
SAIDA_FONTES.write_text(json.dumps(metadados, ensure_ascii=False, indent=2), encoding="utf-8")
print(SAIDA_FONTES.resolve())

/Users/valandro/Downloads/aula03_notebooks/dados/aquisicao_geografica/preparados/porto_alegre/fontes_e_preparo.json


## 10. Reabrir: conferir se os arquivos são reutilizáveis

Salvar sem testar a leitura pode esconder problemas de coluna, codificação ou formato. Reabrimos os três arquivos e comparamos códigos, quantidade de unidades, população e CRS.

O CSV não traz um esquema de tipos: ao reabri-lo, declare a chave como texto.

In [33]:
reaberto = gpd.read_file(SAIDA_GPKG, layer="territorios")
web_reaberto = gpd.read_file(SAIDA_GEOJSON)
tabela_reaberta = pd.read_csv(SAIDA_CSV, dtype={CODIGO: str})
for dados in [reaberto, web_reaberto, tabela_reaberta]:
    assert len(dados) == len(analise)
    assert set(dados[CODIGO]) == set(analise[CODIGO])
    assert int(dados.populacao.sum()) == int(analise.populacao.sum())
assert reaberto.crs == analise.crs
assert web_reaberto.crs.to_epsg() == 4326
print("Leitura validada para GPKG, GeoJSON e CSV.")

Leitura validada para GPKG, GeoJSON e CSV.


## 11. Conferência visual com Folium

Este mapa é criado a partir do **GeoJSON reaberto**, demonstrando que outro notebook pode consumir a saída sem refazer os downloads e as junções. As coordenadas GeoJSON são longitude/latitude; `fit_bounds` do Folium recebe latitude/longitude.

O HTML exportado precisa de internet para o mapa-base e as bibliotecas JavaScript/CSS. Se a saída não aparecer no Jupyter, marque o notebook como confiável (*Trust Notebook*).

In [34]:
mapa = folium.Map(tiles="OpenStreetMap", height=550, control_scale=True)
coropletico = folium.Choropleth(
    geo_data=web_reaberto.__geo_interface__, data=web_reaberto,
    columns=[CODIGO, "densidade_hab_km2"], key_on=f"feature.properties.{CODIGO}",
    fill_color="YlOrRd", bins=6, fill_opacity=0.75,
    line_color="white", line_weight=0.6, highlight=True,
    legend_name=f"{config['nome']} — Censo 2022 — {NIVEL} — hab./km²",
    name="Densidade"
).add_to(mapa)
folium.GeoJsonTooltip(
    fields=[NOME, "populacao", "area_km2", "densidade_hab_km2"],
    aliases=["Unidade:", "População:", "Área (km²):", "Densidade (hab./km²):"],
    localize=True
).add_to(coropletico.geojson)
oeste, sul, leste, norte = web_reaberto.total_bounds
mapa.fit_bounds([[sul, oeste], [norte, leste]])
mapa.save(str(PASTA_SAIDA / "mapa_conferencia.html"))
mapa

## 12. Levar para outro notebook

Os produtos estão em `dados/aquisicao_geografica/preparados/<cidade>/`. Para continuar uma análise, basta ler o GeoPackage:

```python
import geopandas as gpd
territorios = gpd.read_file(
    "dados/aquisicao_geografica/preparados/recife/territorios_indicadores.gpkg",
    layer="territorios"
)
```

O exemplo supõe execução a partir da pasta `notebooks` e que Recife já tenha sido processada. Para compartilhar, copie a pasta da cidade, incluindo `fontes_e_preparo.json`.

### Exercícios

1. Execute o fluxo para Recife e confira códigos e quantidade de bairros.
2. Repita para São Paulo. Explique por que o resultado utiliza distritos.
3. Abra `fontes_e_preparo.json` e identifique o ano dos dados, a edição da tabela e a projeção de área.
4. Compare os tamanhos do GeoPackage e do GeoJSON. Modifique a tolerância de simplificação e observe a mudança nos limites.
5. Escolha outra cidade. Investigue primeiro a disponibilidade de bairros ou distritos; depois acrescente sua configuração.
6. Procure um segundo indicador oficial. Confirme unidade territorial, ano, código e definição antes de fazer a junção.

### Se algo não funcionar

| Situação | O que verificar |
|---|---|
| Download falhou | Conexão, URL e diretório oficial; alternativamente, baixe manualmente no caminho mostrado |
| Município sem registros | Código IBGE, UF e disponibilidade da unidade territorial |
| Junção deixou ausências | Tipo dos códigos, edição da malha e cobertura da tabela |
| População não é numérica | Símbolos de ausência/sigilo e dicionário de dados; não substitua por zero |
| Densidade incoerente | CRS métrico, área em km², ano e cobertura territorial |
| Mapa vazio no navegador | Confiança do notebook, conexão e carregamento de JavaScript |

### Fontes oficiais

- [Malhas de bairros por UF — IBGE](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/bairros/gpkg/UF/)
- [Malhas de distritos por UF — IBGE](https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/malhas_de_setores_censitarios__divisoes_intramunicipais/censo_2022/distritos/gpkg/UF/)
- [Agregados do Censo 2022 e dicionário — IBGE](https://ftp.ibge.gov.br/Censos/Censo_Demografico_2022/Agregados_por_Setores_Censitarios/)
- [ObservaPOA — Mapas](https://prefeitura.poa.br/smpg/observapoa/mapas)
- [São Paulo — limites de subprefeituras e distritos](https://prefeitura.sp.gov.br/web/licenciamento/w/servicos/341586)